In [1]:
# Cell 1
import re
import pandas as pd
from pathlib import Path

SRC_DIR = Path('.')

def simple_norm(s):
    if pd.isna(s): return None
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s else None

print('설정 완료')

설정 완료


In [6]:
# Cell 1
import re
import pandas as pd
from pathlib import Path

SRC_DIR = Path('.')

def simple_norm(s):
    if pd.isna(s): return None
    s = str(s).lower()
    
    return s if s else None

print('설정 완료')

설정 완료


In [7]:
# Cell 2
# names.csv 로드 — 피고/원고 이름 전체
print('names.csv 로드 중...')
names = pd.read_csv(SRC_DIR / 'names.csv', usecols=['name'], low_memory=False)
names['name_norm'] = names['name'].apply(simple_norm)
names = names.dropna(subset=['name_norm'])

name_set = set(names['name_norm'].unique())
print(f'names.csv 고유 이름: {len(name_set):,}개')

names.csv 로드 중...
names.csv 고유 이름: 171,764개


In [8]:
# Cell 3
# g_assignee_disambiguated.tsv 로드
print('g_assignee_disambiguated.tsv 로드 중...')
asgn_list = []
for chunk in pd.read_csv(
    SRC_DIR / 'g_assignee_disambiguated.tsv', sep='	',
    usecols=['disambig_assignee_organization'], chunksize=500_000
):
    normed = chunk['disambig_assignee_organization'].apply(simple_norm).dropna()
    asgn_list.append(normed)

asgn_series = pd.concat(asgn_list).drop_duplicates()
asgn_set = set(asgn_series)
print(f'g_assignee 고유 기관: {len(asgn_set):,}개')

g_assignee_disambiguated.tsv 로드 중...
g_assignee 고유 기관: 516,032개


In [9]:
# Cell 4
# 매칭 확인
matched   = name_set & asgn_set
unmatched = name_set - asgn_set

print(f'=== 매칭 결과 ===')
print(f'names.csv 고유 이름:   {len(name_set):,}')
print(f'g_assignee 고유 기관:  {len(asgn_set):,}')
print(f'매칭:                  {len(matched):,} ({len(matched)/len(name_set):.1%})')
print(f'미매칭:                {len(unmatched):,} ({len(unmatched)/len(name_set):.1%})')
print()

# 매칭된 예시
print('=== 매칭된 이름 샘플 20개 ===')
for n in sorted(matched)[:20]:
    print(f'  {n}')

=== 매칭 결과 ===
names.csv 고유 이름:   171,764
g_assignee 고유 기관:  516,032
매칭:                  19,594 (11.4%)
미매칭:                152,170 (88.6%)

=== 매칭된 이름 샘플 20개 ===
  1 energy solutions, inc.
  1,4group, inc.
  1-800 contacts, inc.
  100% speedlab, llc
  101 holdings
  1061933 ontario inc.
  180s, inc.
  1964 ears, llc
  1994 weyer family limited partnership
  1byone products inc.
  2080 media, inc.
  21 srl
  212, llc
  2201028 ontario inc.
  24/7 customer, inc.
  24/7 media, inc.
  2gig technologies, inc.
  2point communications, inc.
  2wire, inc.
  2xj enterprises, inc.


In [10]:
# Cell 5
# 미매칭 빈도 상위 — 자주 등장하는데 USPTO에 없는 이름
unmatch_freq = (
    names[names['name_norm'].isin(unmatched)]
    .groupby('name_norm').size()
    .sort_values(ascending=False)
    .head(50)
    .reset_index(name='count')
)
print('=== 미매칭 상위 50건 ===')
unmatch_freq

=== 미매칭 상위 50건 ===


,name_norm,count
0,a delaware corporation,8720
1,doing business as,7129
2,a california corporation,6345
3,an individual,3701
4,"blue spike, llc",2082
5,terminated: none,1906
6,does,1594
7,also known as,1520
8,a florida corporation,1474
9,a washington corporation,1092


In [11]:
names[names['name'].str.contains('a delaware corporation', case=False, na=False)]

,name,name_norm
156,a Delaware corporation,a delaware corporation
221,a Delaware Corporation,a delaware corporation
230,a Delaware Corporation,a delaware corporation
233,a Delaware Corporation,a delaware corporation
235,a Delaware Corporation,a delaware corporation
...,...,...
713766,a Delaware corporation,a delaware corporation
714967,"Purple Leaf, LLC, a Delaware corporation","purple leaf, llc, a delaware corporation"
722511,"Procore Technologies, Inc., a Delaware corpora...","procore technologies, inc., a delaware corpora..."
733348,"VIEWPOINT, INC a Delaware corporation","viewpoint, inc a delaware corporation"
